# Dataset Inspection

This notebook combines automated dataset checks with random visual inspection. Run it from the repository root after downloading the dataset and making sure `data/splits.csv` is available.

Automated checks cover image extensions, dimensions, formats, corruption, split/class counts, missing CSV paths, exact duplicates, and duplicate leakage across splits. The visual section samples images from each split and class for manual review.

## 1. Configuration

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import csv
import random

from PIL import Image
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()

if not (REPO_ROOT / "data" / "raw" / "CIFAKE").is_dir():
    REPO_ROOT = REPO_ROOT.parent

ROOT = REPO_ROOT / "data" / "raw" / "CIFAKE"
SPLITS_FILE = REPO_ROOT / "data" / "splits.csv"
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42

random.seed(SEED)

print("Current directory:", Path.cwd())
print("Dataset folder exists:", ROOT.exists())
print("Splits file exists:", SPLITS_FILE.exists())

Current directory: /Users/chengruiyan/Projects/AI-image-detector/notebooks
Dataset folder exists: False
Splits file exists: False


## 2. Automated inspection

This section checks the files physically present under `data/raw/`.

In [ ]:
dimensions = Counter()
formats = Counter()
extensions = Counter()
corrupt = []
duplicate_groups = defaultdict(list)

def file_hash(path):
    digest = hashlib.md5()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in VALID_EXTENSIONS:
        continue

    extensions[path.suffix.lower()] += 1

    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            dimensions[image.size] += 1
            formats[image.format] += 1
        duplicate_groups[file_hash(path)].append(str(path))
    except Exception as error:
        corrupt.append((str(path), str(error)))

print("Total image files:", sum(extensions.values()))
print("\nImage dimensions:")
for size, count in dimensions.most_common():
    print(f"{size}: {count}")
print("\nFile formats:")
for image_format, count in formats.most_common():
    print(f"{image_format}: {count}")
print("\nFile extensions:")
for extension, count in extensions.most_common():
    print(f"{extension}: {count}")
print("\nCorrupt images:", len(corrupt))
for path, error in corrupt[:20]:
    print(path, error)

## 3. Validate `splits.csv`

This checks how the repository intends to use the files: train, validation, and test, with labels REAL = 0 and FAKE = 1.

In [ ]:
split_counts = Counter()
class_counts = Counter()
image_records = []

with SPLITS_FILE.open(newline="") as file:
    rows = list(csv.DictReader(file))

required_columns = {"image_path", "label", "split", "class_name"}
missing_columns = required_columns - set(rows[0].keys()) if rows else required_columns
print("CSV rows:", len(rows))
print("Missing required columns:", missing_columns)

for row in rows:
    split_counts[row["split"]] += 1
    class_counts[(row["split"], row["label"], row["class_name"])] += 1
    image_path = Path(row["image_path"])
    image_records.append({
        "path": str(image_path),
        "split": row["split"],
        "label": row["label"],
    })

print("\nSplit counts:")
for split, count in sorted(split_counts.items()):
    print(f"{split}: {count}")

print("\nClass counts:")
for key, count in sorted(class_counts.items()):
    print(f"{key}: {count}")

missing_paths = [
    record["path"]
    for record in image_records
    if not Path(record["path"]).exists()
]
print("\nMissing paths in splits.csv:", len(missing_paths))
for path in missing_paths[:20]:
    print(path)

## 4. Check duplicates and split leakage

Exact duplicate files are grouped by their MD5 hash. The critical problem is a duplicate appearing in different splits, such as train and test.

In [ ]:
duplicate_groups = {
    digest: paths
    for digest, paths in duplicate_groups.items()
    if len(paths) > 1
}

print("Duplicate groups:", len(duplicate_groups))
for paths in list(duplicate_groups.values())[:10]:
    print(paths)

path_to_split = {
    record["path"]: record["split"]
    for record in image_records
}

cross_split_duplicates = []
for paths in duplicate_groups.values():
    splits = {
        path_to_split[path]
        for path in paths
        if path in path_to_split
    }
    if len(splits) > 1:
        cross_split_duplicates.append((paths, splits))

print("\nDuplicate groups crossing splits:", len(cross_split_duplicates))
for paths, splits in cross_split_duplicates[:10]:
    print("Splits:", splits)
    print(paths)

## 5. Random visual inspection

The automated checks cannot detect every label error or visual shortcut. This section displays a stratified random sample from each split and class. Inspect whether the labels look plausible and whether one class has an obvious source, style, colour, watermark, or quality advantage.

In [ ]:
def show_random_samples(split, class_name, sample_count=12):
    folder = ROOT / split / class_name
    image_paths = [
        path for path in folder.glob("*")
        if path.is_file()
        and path.suffix.lower() in VALID_EXTENSIONS
    ]

    if not image_paths:
        print(f"No images found for {split}/{class_name}: {folder}")
        return

    sample_paths = random.sample(
        image_paths,
        min(sample_count, len(image_paths)),
    )

    columns = 4
    rows = (len(sample_paths) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows))
    axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
    fig.suptitle(f"{split.upper()} - {class_name}", fontsize=16)

    for path, axis in zip(sample_paths, axes):
        try:
            with Image.open(path) as image:
                axis.imshow(image.convert("RGB"))
            axis.set_title(path.name[:18])
        except Exception as error:
            axis.set_title("Could not open")
            print(path, error)
        axis.axis("off")

    for axis in axes[len(sample_paths):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()

for split in ["train", "validation", "test"]:
    for class_name in ["REAL", "FAKE"]:
        show_random_samples(split, class_name)

## 6. Findings and decisions

Complete this section after reviewing the outputs. Do not describe visual observations as automated results.

- Corrupt images: ___
- Missing CSV paths: ___
- Exact duplicate groups: ___
- Duplicate groups crossing splits: ___
- Are REAL and FAKE reasonably balanced? ___
- Are there obvious watermarks or class-specific visual shortcuts? ___
- Are any labels suspicious? ___
- Action taken before training: ___